# Agent Company — 交互式教程

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/panda/agent-company/blob/main/notebooks/interactive_demo.ipynb)

本教程将带你逐步体验 Agent Company 框架的核心功能：

1. **人才池** — 探索预设的 AI Agent 库
2. **需求分析** — 将自然语言需求转化为结构化招标书
3. **招标组建** — 从人才池筛选、竞标、评审、组队
4. **价值观系统** — 29 条价值观准则 + 审计检测
5. **模型经济** — S/A/B/C 四级模型 + 预算策略
6. **绩效评审** — KPI 追踪 + 定期评审 + 末位淘汰
7. **健康度监控** — 十二维健康评估
8. **完整流程** — 端到端演示

> 本教程使用模拟数据，无需任何 API key。

## 1. 安装

In [ ]:
# 从 PyPI 安装（发布后）
# !pip install agent-company

# 或从 GitHub 源码安装
# !pip install git+https://github.com/panda/agent-company.git#subdirectory=packages/core

# 本地开发时，添加路径即可
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'packages', 'core', 'src'))
# 如果在 Colab 上，取消上面 pip install 的注释

print('安装完成!')

## 2. 导入核心模块

In [ ]:
from agent_company.pool.presets import create_default_pool
from agent_company.pool.talent_pool import TalentPool
from agent_company.tender.analyzer import RequirementAnalyzer, TenderSpec, RoleSpec
from agent_company.tender.engine import TenderEngine, TenderResult
from agent_company.values.vault import ValueVault, ValuePrinciple
from agent_company.values.matcher import ValueMatcher
from agent_company.values.system import ValueSystem
from agent_company.economy.budget import ModelBudgetManager, BudgetStrategy, CostRecord
from agent_company.economy.model_tiers import ModelTierRegistry
from agent_company.performance.engine import PerformanceEngine
from agent_company.performance.elimination import EliminationEngine
from agent_company.health.monitor import HealthMonitor

import random
random.seed(42)

print('所有模块导入成功!')

## 3. 探索人才池

Agent Company 内置了一个预设人才池，包含多种角色的 AI Agent：写作、工程、设计、分析等。

In [ ]:
# 创建默认人才池
pool = create_default_pool()
print(f'人才池规模: {pool.size} 个 Agent\n')

# 按类别查询
print('=== 写作类 Agent ===')
writers = pool.query(role_match='writer', limit=5)
for agent in writers:
    skills = list(agent.skills.keys())[:4]
    print(f'  {agent.name:12s} | 等级: {agent.model_tier} | 技能: {skills}')

print('\n=== 工程类 Agent ===')
engineers = pool.query(role_match='engineer', limit=5)
for agent in engineers:
    skills = list(agent.skills.keys())[:4]
    print(f'  {agent.name:12s} | 等级: {agent.model_tier} | 技能: {skills}')

# 查看单个 Agent 详情
print('\n=== Agent 详情示例 ===')
sample = pool.query(limit=1)[0]
print(f'  ID: {sample.id}')
print(f'  名称: {sample.name}')
print(f'  分类: {sample.category}')
print(f'  模型等级: {sample.model_tier}')
print(f'  技能: {dict(list(sample.skills.items())[:5])}')

## 4. 需求分析

`RequirementAnalyzer` 将自然语言需求转化为结构化的招标书 (`TenderSpec`)，包括项目类型、所需角色、质量标准等。

In [ ]:
analyzer = RequirementAnalyzer()

# 分析一个内容创作需求
user_request = '写一篇关于 AI Agent 多智能体协作的技术博客，涵盖架构设计、招标制和绩效管理'
spec = analyzer.analyze(user_request, budget=10.0)

print(f'用户需求: {user_request}')
print(f'\n━━━ 分析结果 ━━━')
print(f'项目类型: {spec.project_type}')
print(f'复杂度: {spec.estimated_complexity}')
print(f'预算: ${spec.budget_usd}')
print(f'\n交付物:')
for d in spec.deliverables:
    print(f'  - {d}')
print(f'\n质量标准:')
for k, v in spec.quality_standards.items():
    print(f'  {k}: {v}')
print(f'\n所需角色:')
for role in spec.required_roles:
    print(f'  {role.name} x{role.count} [{role.priority}]')
    print(f'    必备技能: {role.must_have_skills}')
    print(f'    最低模型等级: {role.min_model_tier}')

## 5. 运行招标

`TenderEngine` 执行完整的招标流程：初筛 → 竞标 → 评审打分 → 团队组建。

In [ ]:
engine = TenderEngine(pool)
result = engine.run_tender(spec)

# 招标日志
print('=== 招标过程日志 ===')
for entry in result.tender_log:
    print(f'  {entry}')

# 中标团队
print(f'\n=== 中标团队 ({len(result.selected_team)} 人) ===')
for member in result.selected_team:
    p = member['profile']
    r = member['role']
    s = member['score']
    print(f'  {p.name:12s} -> {r.name:6s} | 模型: {p.model_tier} | 得分: {s.total_score:.1f}')

print(f'\n公司名称: {result.company.name}')
print(f'落选人数: {len(result.rejected)}')

## 6. 价值观系统

Agent Company 内置了来自顶级企业的 29 条价值观准则，分为 7 大类别。
价值观系统可以：
- 将价值观转化为具体的行为约束 Prompt
- 审计消息/交付物是否符合价值观

In [ ]:
# 加载价值观库
vault = ValueVault()
all_values = vault.all()
print(f'价值观库共 {len(all_values)} 条准则\n')

# 按分类展示
from collections import Counter
categories = Counter(v.category for v in all_values)
print('分类统计:')
for cat, count in categories.most_common():
    print(f'  {cat:18s} {count} 条')

# 展示几条示例
print('\n示例价值观:')
for v in all_values[:3]:
    print(f'  [{v.category}] {v.name}')
    print(f'    来源: {v.origin}')
    print(f'    准则: {v.rule[:60]}...')
    print(f'    违反: {v.violation[:60]}...')
    print()

In [ ]:
# 价值观匹配 + 注入
matcher = ValueMatcher(vault)
matched = matcher.match_for_task('creative_writing')
print(f'创意写作任务匹配了 {len(matched)} 条价值观\n')

# 生成行为约束 Prompt
value_system = ValueSystem(matched)
prompt = value_system.generate_behavior_prompt()
print('=== 行为约束 Prompt (预览) ===')
print(prompt[:600])
print(f'\n... (共 {len(prompt)} 字符)')

In [ ]:
# 价值观审计演示
print('=== 价值观审计 ===')
test_messages = [
    '这篇文章差不多就行了，赶紧发吧',
    '我已完成第三章初稿，质量评分 92 分，事实核查全部通过，交付结果如附件所示',
    '这个不是我的问题，让其他人处理吧',
    '大概是可以的，不太确定具体时间',
]

for msg in test_messages:
    audit = value_system.audit_message(msg)
    status = 'PASS' if audit.score >= 0.8 else 'WARN' if audit.score >= 0.5 else 'FAIL'
    print(f'\n[{status}] 分数: {audit.score:.2f}')
    print(f'  消息: "{msg}"')
    if audit.violations:
        print(f'  违反: {audit.violations[0]}')
    if audit.suggestions:
        print(f'  建议: {audit.suggestions[0]}')

## 7. 模型经济

Agent Company 把模型调用当作「薪资」管理：
- **S 级** (Claude Opus, GPT-4o): 顶级岗位，如 CEO、主编
- **A 级** (Claude Sonnet, GPT-4o-mini): 高级岗位，如项目经理
- **B 级** (Claude Haiku, Gemini Flash): 常规岗位，如初级工程师
- **C 级** (轻量模型): 基础岗位，如校对、数据录入

In [ ]:
# 模型等级注册表
registry = ModelTierRegistry()
print('=== 模型等级体系 ===')
for tier in ['S', 'A', 'B', 'C']:
    models = registry.get_models_by_tier(tier)
    print(f'\n{tier} 级模型:')
    for m in models:
        print(f'  {m.id:25s} | 能力: {m.capability:3.0f} | '
              f'成本: ${m.cost_per_1k_input:.4f}/1k入 ${m.cost_per_1k_output:.4f}/1k出')
        print(f'  {"": <25s} | 优势: {", ".join(m.strengths[:3])}')

In [ ]:
# 预算管理演示
budget_mgr = ModelBudgetManager(total_budget=10.0, strategy=BudgetStrategy.BALANCED)
print(f'策略: {budget_mgr.strategy.value}')
print(f'总预算: ${budget_mgr.total_budget}')

# 为角色分配模型
roles = [
    {'role': '主编', 'priority': 'critical'},
    {'role': '作者', 'priority': 'core'},
    {'role': '校对', 'priority': 'support'},
]
allocation = budget_mgr.allocate_models(roles, registry)
print(f'\n模型分配结果:')
for role_name, model_id in allocation.items():
    print(f'  {role_name} -> {model_id}')

# 模拟一些调用记录
budget_mgr.record_cost(CostRecord(agent_id='agent-1', model_id='claude-opus-4-6',
                                    input_tokens=2000, output_tokens=1000, cost_usd=0.105))
budget_mgr.record_cost(CostRecord(agent_id='agent-2', model_id='claude-sonnet-4-6',
                                    input_tokens=1500, output_tokens=800, cost_usd=0.0165))
budget_mgr.record_cost(CostRecord(agent_id='agent-3', model_id='claude-haiku',
                                    input_tokens=1000, output_tokens=500, cost_usd=0.0005))

report = budget_mgr.get_budget_report()
print(f'\n预算报告:')
print(f'  已花费: ${report["spent"]:.4f}')
print(f'  剩余: ${report["remaining"]:.4f}')
print(f'  使用率: {report["usage_ratio"]*100:.1f}%')
print(f'  调用次数: {report["total_records"]}')

## 8. 绩效评审

`PerformanceEngine` 实时追踪 Agent 的 KPI，并定期执行评审：
- S/A/B/C/D/F 六级评分
- 排名、警告、淘汰机制
- 末位淘汰 + 人才池补充

In [ ]:
# 初始化绩效引擎
perf_engine = PerformanceEngine(
    review_interval=5,
    elimination_threshold=50.0,
    warning_threshold=65.0,
)

# 注册团队成员
team_members = [
    ('editor-01', '主编', 'Alice'),
    ('writer-01', '作者', 'Bob'),
    ('writer-02', '作者', 'Charlie'),
    ('proofer-01', '校对', 'Diana'),
]

for agent_id, role, name in team_members:
    perf_engine.register_agent(agent_id, role, name)
print(f'已注册 {len(team_members)} 名成员\n')

# 模拟工作信号
performance_profiles = {
    'editor-01': {'quality': 0.92, 'speed': 2.0, 'on_time': True},   # 优秀
    'writer-01': {'quality': 0.78, 'speed': 3.5, 'on_time': True},   # 良好
    'writer-02': {'quality': 0.65, 'speed': 5.0, 'on_time': False},  # 一般
    'proofer-01': {'quality': 0.35, 'speed': 12.0, 'on_time': False}, # 差
}

for tick in range(5):
    for agent_id, profile in performance_profiles.items():
        perf_engine.on_message(agent_id, profile['speed'] + random.uniform(-0.5, 0.5),
                              int(profile['quality'] * 2000), tick=tick)
        if tick % 2 == 0:
            perf_engine.on_task_complete(agent_id, profile['quality'], profile['on_time'], tick=tick)

# 执行评审
agent_ids = [m[0] for m in team_members]
review = perf_engine.periodic_review(agent_ids)

print(f'=== 绩效评审结果 ===')
print(f'公司整体得分: {review.company_score:.1f}\n')
print(f'{"#":>2} {"\u59d3\u540d":<10} {"\u89d2\u8272":<6} {"\u5f97\u5206":>6} {"\u7b49\u7ea7":>4} {"\u8d8b\u52bf"}')
print('-' * 45)
for ps in review.rankings:
    print(f'{ps.rank:>2} {ps.agent_name:<10} {ps.role:<6} {ps.total_score:>6.1f} {ps.grade.value:>4} {ps.trend}')

if review.warnings:
    print(f'\n⚠️ 警告: {[ps.agent_name for ps in review.warnings]}')
if review.eliminations:
    print(f'❌ 淘汰: {[ps.agent_name for ps in review.eliminations]}')

In [ ]:
# 末位淘汰演示
elimination_engine = EliminationEngine(pool, bottom_ratio=0.2)

replacements = elimination_engine.process_eliminations(
    review=review,
    company_values=['quality', 'creativity'],
    current_agent_ids=agent_ids,
)

print('=== 末位淘汰结果 ===')
if replacements:
    for r in replacements:
        print(f'\n  淘汰: {r.removed_name} (得分: {r.score:.1f}, 等级: {r.grade})')
        print(f'  原因: {r.reason}')
        print(f'  替补: {r.replaced_by_name}')
else:
    print('  本轮无需淘汰')

## 9. 健康度监控

`HealthMonitor` 从 12 个维度评估公司健康状态：
组织、社会、业务、心理、伦理、生态、信息、文化、政治、时间、经济、学习

In [ ]:
monitor = HealthMonitor()

# 构建公司数据
company_data = {
    'agents': [
        {'id': 'editor-01', 'name': 'Alice', 'role': '主编', 'workload': 0.7},
        {'id': 'writer-01', 'name': 'Bob', 'role': '作者', 'workload': 0.8},
        {'id': 'writer-02', 'name': 'Charlie', 'role': '作者', 'workload': 0.6},
        {'id': 'proofer-01', 'name': 'Diana', 'role': '校对', 'workload': 0.9},
    ],
    'departments': [{'name': '编辑部', 'size': 4}],
    'messages': [
        {'sender': 'editor-01', 'channel': 'general', 'length': 200},
        {'sender': 'writer-01', 'channel': 'general', 'length': 150},
        {'sender': 'writer-02', 'channel': 'general', 'length': 100},
    ],
    'tasks': [
        {'status': 'completed', 'quality': 0.85},
        {'status': 'in_progress', 'quality': 0.0},
    ],
    'performance_scores': {ps.agent_id: ps.total_score for ps in review.rankings},
    'budget_info': {'total': 10.0, 'spent': 3.5, 'remaining': 6.5},
    'governance_rules': ['hierarchical', 'voting'],
    'values': ['quality', 'creativity', 'collaboration'],
}

# 执行评估
report = monitor.evaluate(company_data)

# 打印报告
print(monitor.print_report(report))

## 10. 完整流程：端到端演示

下面将上述所有步骤串联起来，展示一个完整的工作循环。

In [ ]:
import time

start = time.time()
print('=' * 60)
print('  Agent Company — 端到端完整流程')
print('=' * 60)

# 1. 需求
request = '开发一个 AI 驱动的客户服务聊天机器人，支持多语言和情感分析'
print(f'\n[1] 需求: {request}')
spec = RequirementAnalyzer().analyze(request, budget=20.0)
print(f'    → 项目类型: {spec.project_type}, 角色: {len(spec.required_roles)} 个')

# 2. 招标
pool = create_default_pool()
result = TenderEngine(pool).run_tender(spec)
print(f'\n[2] 招标: 从 {pool.size} 人中选出 {len(result.selected_team)} 人')
for m in result.selected_team:
    print(f'    {m["profile"].name} -> {m["role"].name} (得分: {m["score"].total_score:.1f})')

# 3. 价值观
vault = ValueVault()
matched = ValueMatcher(vault).match_for_task('software_development')
vs = ValueSystem(matched)
print(f'\n[3] 价值观: {len(matched)} 条准则已注入')

# 4. 预算
budget = ModelBudgetManager(total_budget=20.0, strategy=BudgetStrategy.BALANCED)
roles_for_budget = [{'role': m['role'].name, 'priority': m['role'].priority}
                    for m in result.selected_team]
model_alloc = budget.allocate_models(roles_for_budget, ModelTierRegistry())
print(f'\n[4] 模型分配:')
for role, model in model_alloc.items():
    print(f'    {role} -> {model}')

# 5. 模拟执行 + 绩效
perf = PerformanceEngine(review_interval=5)
ids = []
for m in result.selected_team:
    p = m['profile']
    perf.register_agent(p.id, m['role'].name, p.name)
    ids.append(p.id)
    # 模拟工作
    for tick in range(5):
        q = random.uniform(0.6, 0.95)
        perf.on_message(p.id, random.uniform(1, 5), int(q * 2000), tick=tick)
        perf.on_task_complete(p.id, q, random.random() > 0.2, tick=tick)

review = perf.periodic_review(ids)
print(f'\n[5] 绩效评审: 公司得分 {review.company_score:.1f}')
for ps in review.rankings:
    print(f'    #{ps.rank} {ps.agent_name:12s} {ps.total_score:.1f} ({ps.grade.value})')

# 6. 健康度
health = HealthMonitor().evaluate({
    'agents': [{'id': m['profile'].id, 'name': m['profile'].name, 'role': m['role'].name}
              for m in result.selected_team],
    'departments': [{'name': '工程部', 'size': len(result.selected_team)}],
    'messages': [],
    'tasks': [{'status': 'completed', 'quality': 0.8}],
    'performance_scores': {ps.agent_id: ps.total_score for ps in review.rankings},
    'budget_info': budget.get_budget_report(),
    'governance_rules': ['hierarchical'],
    'values': spec.value_priorities,
})
print(f'\n[6] 健康度: {health.overall_score:.1f}/100 (等级: {health.grade})')

elapsed = time.time() - start
print(f'\n{"=" * 60}')
print(f'  完成! 总耗时: {elapsed:.2f} 秒')
print(f'  团队: {len(result.selected_team)} 人 | 绩效: {review.company_score:.1f} | 健康: {health.overall_score:.1f}')
print('=' * 60)